# Chapter 03-05 · Bayes' rule you can do on paper

**Label:** Core  |  **Time:** ~45 minutes  |  **Difficulty:** gentle if you do it by counting, which
is how this chapter does it

**Prerequisites:** 03-04. You should be able to read a conditional probability off a table and say
why `P(A | B)` and `P(B | A)` differ.

**Position in the learning path:** module 03, chapter 5 of 8. Before: **03-04**. After: **03-06**,
which turns to functions, lines and logarithms.

---

## Why this matters

03-04 computed `P(cracked | alarm)` from a table of counts. Real problems rarely hand you the table.
They hand you **rates**:

> "The condition affects 1 in 2,000 people. The test detects 99% of cases and gives a false positive
> 1% of the time. Your result is positive."

There is no table. There is a prior belief, some evidence, and a need to revise. Bayes' rule is the
arithmetic for that revision, and it is genuinely doable on paper - the version most people find
impossible is impossible only because it is written in the wrong form.

The chapter ends somewhere less comfortable. Bayes' rule is exactly right, and the standard way of
applying it twice gives **83%** on a problem whose true answer is **5%** - not because the rule
fails, but because of an assumption smuggled in beside it.

## What you will be able to do

- Turn any set of rates into a table of counts, and read the answer off it
- Write Bayes' rule, and recognise it as the counting you already did
- Use the odds form to update beliefs in your head
- Update on several pieces of evidence in sequence, and say when you may not
- Explain why two positive tests can mean much less than they appear to

## Warm-up: retrieve, do not reread

1. Why was `P(cracked | alarm)` only 0.27 when the sensor caught 90% of cracks?
2. What does independence mean in terms of counts in a table?
3. Two cables each fail 1% of the time. Why was "both fail" 1 in 247 rather than 1 in 10,040?

<br>

*Answers: (1) there were 960 sound bikes to raise false alarms about and only 40 cracked ones, so
false alarms outnumbered true ones 96 to 36. (2) every cell equals row total times column total over
grand total. (3) they came from the same batch - the first failure is evidence that this bike's batch
is bad, so the second is far more likely to fail than its marginal rate suggests.*

## The method: stop using probabilities

The single most effective trick in this entire chapter is to **refuse to work in percentages**.
Invent a population, turn every rate into a count of people, and then count.

The rates: 1 in 2,000 have the condition, the test detects 99% of cases, and it is positive for 1%
of people who do not have it. Take 100,000 people.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

population = 100_000
base_rate = 1 / 2000
sensitivity = 0.99          # P(positive | has it)
false_positive_rate = 0.01  # P(positive | does not have it)

has_it = population * base_rate
does_not = population - has_it

table = pd.DataFrame(
    {"test positive": [has_it * sensitivity, does_not * false_positive_rate],
     "test negative": [has_it * (1 - sensitivity), does_not * (1 - false_positive_rate)]},
    index=["has the condition", "does not"])
table.index.name = "out of 100,000 people"

print(table.round(1).to_string())
print()
print("of the %.0f positive results, %.1f are real" % (table["test positive"].sum(),
                                                       table.loc["has the condition", "test positive"]))
print("P(has it | positive) = %.1f / %.1f = %.4f  = %.2f%%"
      % (table.loc["has the condition", "test positive"], table["test positive"].sum(),
         table.loc["has the condition", "test positive"] / table["test positive"].sum(),
         100 * table.loc["has the condition", "test positive"] / table["test positive"].sum()))

### 4.72%

**Fifty people have it and forty-nine are found. Nine hundred and ninety-nine healthy people are told
they might.** The positive result multiplied the chance by about 94 - from 1 in 2,000 to 1 in 21 -
and 1 in 21 is still mostly "no".

That paragraph is the whole of Bayes' rule, and it was done with multiplication and one division. The
step people find hard is not the arithmetic; it is remembering that **99% describes what the test
does to people who have the condition, and there are hardly any of those**.

**This is the method to use for the rest of your life:** given rates, invent a population large
enough to make the counts whole, fill in four cells, divide. It is faster than the formula, it is
harder to get backwards, and it can be done on the back of a letter while the person who received the
letter is watching.

## The same thing, written as a formula

Now name what was just done. Writing `H` for the hypothesis (has the condition) and `E` for the
evidence (tested positive):

> `P(H | E) = P(E | H) x P(H) / P(E)`

Each piece is one of the numbers above:

| Name | Meaning here | Value |
|---|---|---|
| `P(H)` - the **prior** | what you believed before the test | 0.0005 |
| `P(E \| H)` - the **likelihood** | how well the evidence is explained if H is true | 0.99 |
| `P(E)` - the **evidence** | how often this evidence happens at all | 0.01049 |
| `P(H \| E)` - the **posterior** | what you believe after | 0.0472 |

The denominator is the one people forget, and it is just the total positive column: everybody who
tests positive, whether or not they have the condition.

> `P(E) = P(E | H) x P(H) + P(E | not H) x P(not H)`

In [ ]:
prior = base_rate
likelihood = sensitivity
evidence = sensitivity * base_rate + false_positive_rate * (1 - base_rate)
posterior = likelihood * prior / evidence

print("prior      P(H)      = %.6f" % prior)
print("likelihood P(E|H)    = %.6f" % likelihood)
print("evidence   P(E)      = %.6f   (= %.6f x %.6f + %.6f x %.6f)"
      % (evidence, sensitivity, base_rate, false_positive_rate, 1 - base_rate))
print("posterior  P(H|E)    = %.6f" % posterior)
print()
print("by counting, above  : %.6f" % (table.loc["has the condition", "test positive"]
                                      / table["test positive"].sum()))

## The form that fits in your head: odds

Percentages are awkward to update. **Odds are not**, and in odds form Bayes' rule becomes a single
multiplication.

> **posterior odds = prior odds x likelihood ratio**

where the likelihood ratio is `P(E | H) / P(E | not H)` - how much more often this evidence shows up
when the hypothesis is true than when it is false.

For our test: `0.99 / 0.01 = 99`. A positive result is 99 times more likely in someone who has the
condition. So the odds go from 1-to-1999 up to 99-to-1999, which is about 1-to-20.

In [ ]:
def odds_update(prior_odds, likelihood_ratio):
    return prior_odds * likelihood_ratio


def odds_to_probability(odds):
    return odds / (1 + odds)


prior_odds = base_rate / (1 - base_rate)
lr_positive = sensitivity / false_positive_rate
lr_negative = (1 - sensitivity) / (1 - false_positive_rate)

print("prior odds        : %.6f   (1 to %.0f)" % (prior_odds, 1 / prior_odds))
print("likelihood ratio  : %.1f for a positive, %.4f for a negative" % (lr_positive, lr_negative))
print()
after_positive = odds_update(prior_odds, lr_positive)
after_negative = odds_update(prior_odds, lr_negative)
print("after a positive  : odds %.4f  ->  p = %.4f  (%.2f%%)"
      % (after_positive, odds_to_probability(after_positive), 100 * odds_to_probability(after_positive)))
print("after a negative  : odds %.8f  ->  p = %.7f  (1 in %.0f)"
      % (after_negative, odds_to_probability(after_negative),
         1 / odds_to_probability(after_negative)))

### Why this form is worth the five minutes

**"Multiply the odds by 99"** is something you can do while someone is still speaking. The percentage
form is not.

It also makes the asymmetry visible. A positive multiplies the odds by 99; a negative multiplies them
by 0.0101, taking a 1-in-2,000 chance down to **1 in 197,902**. For a rare condition the negative
result is enormously more informative than the positive one - which is exactly what E4 of the last
chapter found by counting the quiet bikes.

A rough scale worth carrying:

| Likelihood ratio | What the evidence does |
|---|---|
| 1 | nothing at all |
| 2 to 5 | mild - worth noting, not worth acting on alone |
| 10 | strong |
| 100 | very strong |
| above 1000 | decisive, and worth double-checking the error rates that produced it |

## Updating twice: yesterday's posterior is today's prior

The clinic repeats the test. It comes back positive again.

In odds form this is trivial - multiply by 99 a second time. That is the property that makes the odds
form worth knowing: **evidence accumulates by multiplication**, and today's posterior odds are
tomorrow's prior odds.

In [ ]:
rows = []
odds = prior_odds
for n_positives in range(0, 4):
    rows.append({"positive tests": n_positives,
                 "odds": round(float(odds), 4),
                 "probability": round(float(odds_to_probability(odds)), 4),
                 "as a percentage": "%.2f%%" % (100 * odds_to_probability(odds))})
    odds = odds_update(odds, lr_positive)

print(pd.DataFrame(rows).to_string(index=False))

One positive: **4.72%**. Two: **83.06%**. Three: **99.79%**.

That progression is why confirmatory testing exists, and it looks like a complete answer to the
problem the first section raised. A single positive is weak evidence about a rare condition; repeat
the test and the doubt disappears.

**It is also, in a way that is invisible from the arithmetic, sometimes completely wrong.**

## Failure lab: the second test is not a second opinion

Multiplying by 99 twice assumes the two results are **independent given the truth** - that whether
the test errs on the second run has nothing to do with whether it erred on the first.

Ask why a test gives a false positive. Two possibilities, both realistic:

- **Model A - the error is noise.** Contamination, a bubble, a machine tolerance, a technician's
  hand. A fresh throw of the dice each time.
- **Model B - the error is the person.** Something stable about them - a cross-reacting protein, a
  benign variant, an unusual baseline - makes this test read positive for them, every time.

**Both models produce exactly the same false-positive rate of 1%.** No amount of measuring the test's
error rate distinguishes them. Simulate both, forty million people each, and ask what two positives
mean.

In [ ]:
sim = np.random.default_rng(0)
n_people = 40_000_000

truly_has_it = sim.random(n_people) < base_rate

# MODEL A: each test errs independently.
a_first = np.where(truly_has_it, sim.random(n_people) < sensitivity,
                   sim.random(n_people) < false_positive_rate)
a_second = np.where(truly_has_it, sim.random(n_people) < sensitivity,
                    sim.random(n_people) < false_positive_rate)

# MODEL B: 1% of healthy people carry a stable trait that always reads positive.
carries_trait = sim.random(n_people) < false_positive_rate
b_first = np.where(truly_has_it, sim.random(n_people) < sensitivity, carries_trait)
b_second = np.where(truly_has_it, sim.random(n_people) < sensitivity, carries_trait)

for label, first, second in [("A - independent errors", a_first, a_second),
                             ("B - a stable trait     ", b_first, b_second)]:
    both = first & second
    print("%s  P(has it | 1 positive) = %.4f   P(has it | 2 positives) = %.4f"
          % (label, (truly_has_it & first).sum() / first.sum(),
             (truly_has_it & both).sum() / both.sum()))

print()
print("the sequential odds calculation says                        : %.4f"
      % odds_to_probability(prior_odds * lr_positive ** 2))

### Diagnosis: 83% or 5%, and the error rates cannot tell you which

| | after one positive | after two positives |
|---|---|---|
| **Model A** - errors are noise | 0.0472 | **0.8308** |
| **Model B** - errors are the person | 0.0472 | **0.0468** |
| the sequential calculation | 0.0472 | 0.8306 |

Under model B the second test adds **nothing whatsoever** - the posterior moves from 4.72% to 4.68%,
which is noise. Anyone who carries the trait tests positive forever, so repeating the test just asks
the same question of the same protein and receives the same answer.

Yet the sequential calculation returns 83.06% in both worlds, because it never asked *why* the test
errs. It knew the false-positive rate, and the false-positive rate is identical in the two models.

**What this is, precisely.** Bayes' rule is not wrong here - it is not even involved. The mistake is
in the likelihood: `P(two positives | healthy)` is `0.01 x 0.01 = 0.0001` under model A and
`0.01` under model B, a hundredfold difference. This is 03-04's brake cables exactly - two events
sharing a hidden common cause, multiplied as though they were independent, with the error always in
the optimistic direction.

**What to do about it, in order of usefulness:**

1. **A second test should be a *different* test**, ideally one whose errors have a different cause.
   That is what a confirmatory test is for, and it is why it is a different assay rather than a rerun.
2. **Ask what would make this evidence wrong**, and whether that thing would also make the next piece
   of evidence wrong. If yes, the second piece is worth far less than its likelihood ratio claims.
3. **Be suspicious of accumulated evidence from one source.** Ten readings from one instrument, ten
   reviews from one community, ten features from one broken sensor: the tenth adds much less than the
   first, and multiplying likelihood ratios ten times says otherwise.

The pattern generalises well beyond testing. **Correlated evidence is the most common way that a
correct rule produces a confident wrong answer**, and it is invisible if you only look at error rates.

## Where the prior comes from, and how much it matters

Every calculation here began with 1 in 2,000, and nothing in the data supplied it. It came from
outside - from a registry, a study, a population statistic.

Since it drives the answer, it is worth knowing how hard it drives.

In [ ]:
rows = []
for described, rate in [("general population screening", 1 / 2000),
                        ("has some symptoms", 1 / 100),
                        ("referred by a specialist", 1 / 10),
                        ("strong family history and symptoms", 1 / 3)]:
    odds = (rate / (1 - rate)) * lr_positive
    rows.append({"who is being tested": described,
                 "prior": "%.4f" % rate,
                 "after one positive": "%.3f" % odds_to_probability(odds)})
print(pd.DataFrame(rows).to_string(index=False))

The same test and the same positive result mean **4.7%** for someone screened at random and **91.7%**
for someone a specialist referred - rising to **98.0%** where there is a family history too.

Two consequences, and they are the practical heart of this chapter:

- **Screening a whole population is a different activity from testing someone who has symptoms**, even
  with identical equipment. This is why mass screening for rare conditions is contentious: the
  arithmetic guarantees that most positives will be false, and the harms of investigating them are
  real.
- **A prior is a claim, and it should be stated.** "1 in 2,000" is not neutral background - it is the
  input doing most of the work. When someone objects that priors are subjective, the honest reply is
  that the prior is *explicit* here, whereas an analysis that skips it has assumed one silently,
  usually 50-50.

### And the reassuring part

Evidence overwhelms a bad prior, if there is enough of it and it is independent. A likelihood ratio
of 99 applied three times moves a 1-in-2,000 prior to 99.8%. Priors matter most when evidence is
weak - which is exactly when people are most tempted to ignore them.

## Common misconceptions

**"Bayes' rule is a formula to memorise."**
It is the counting from 03-04 with names attached. If you can build a four-cell table from rates, you
can do every problem in this chapter without writing the formula down once.

**"A 99% accurate test means a positive is 99% likely to be right."**
It means 4.7% at a base rate of 1 in 2,000, and 97% in a specialist clinic. The test statistic and
the answer to your question are different numbers.

**"The prior is subjective, so Bayesian reasoning is unscientific."**
The prior is *explicit*. An analysis that ignores base rates has not avoided a prior; it has silently
assumed one. Here, "the test says positive so it is probably true" is the assumption that the prior
was around 50%, which was wrong by a factor of a thousand.

**"Repeat the test to be sure."**
Only if the second test's errors are independent of the first's. When they are not - and often they
are not, because false positives frequently have stable causes - the second result adds almost
nothing while the arithmetic claims it adds a factor of 99.

**"Evidence accumulates, so more data always sharpens the answer."**
Independent evidence accumulates. Correlated evidence mostly repeats itself, and treating it as
independent produces confident wrong answers at a rate that grows with how much of it you have.

**"A negative result is less informative than a positive one."**
For a rare condition the reverse is true. Here a negative takes 1 in 2,000 down to 1 in 197,902,
while a positive only reaches 1 in 21.

## Exercises

Solutions: `solutions/03_math_foundations/03-05_bayes_solutions.ipynb`.

### Quick understanding

**E1.** Name the prior, the likelihood and the posterior in the screening problem, and say which one
the test manufacturer can tell you and which one they cannot.

**E2.** Why is the odds form easier to update than the probability form?

**E3.** In one sentence, what has to be true about two tests before you may multiply their likelihood
ratios?

### Hand calculation

**E4.** A condition affects 1 in 100. A test has an 80% detection rate and a 5% false-positive rate.
Using 10,000 people, build the table and compute the chance a positive is real. Then do it again with
the odds form and check you agree.

**E5.** Using the chapter's test (LR 99 for positive, 0.0101 for negative), start from a prior of 1
in 2,000 and update on this sequence: positive, positive, negative. What is the final probability?
Does the order matter? Show why in one line.

**E6.** How large must the likelihood ratio of a single test be for one positive result to take a
1-in-2,000 prior past 50%? Past 95%?

### Coding

**E7.** Write `update(prior, likelihood_ratios)` that takes a prior probability and a list of
likelihood ratios and returns the posterior. Use it to reproduce the chapter's three-positive
sequence, and to check E5.

**E8.** Reproduce the chapter's model A / model B simulation, but vary the share of healthy people
carrying the stable trait from 0% to 1% of the population while holding the overall false-positive
rate at 1%. Plot `P(has it | two positives)` against that share. What shape is it?

**E9.** A spam filter looks at three words, with likelihood ratios 8, 5 and 0.3. The prior probability
that a message is spam is 0.4. Compute the posterior. Then compute it again for a message containing
the first word **three times**, treating each occurrence as independent evidence, and say why the
answer is not credible.

### Interpretation

**E10.** A model outputs "87% probability of churn" for a customer. What would have to be true for
that number to mean what a probability should mean, and how would you check it on historical data?

**E11.** A security scanner flags 1 in 400 passengers. Investigating a flag costs 4 minutes. The base
rate of an actual threat is 1 in 5,000,000. Compute the posterior for a flagged passenger, and say
what the number implies about how the system should be judged.

### Debugging

**E12.** An analyst builds a naive Bayes spam classifier and finds it outputs probabilities of
0.99999 or 0.00001 and almost nothing between. Explain the cause using this chapter's ideas, and name
the assumption in "naive" Bayes that produces it.

### Exam and interview reasoning

**E13.** "A test is 95% accurate and you test positive for a disease affecting 1 in 1,000. Should you
worry?" Answer as you would in an interview: the number, the method you used, and the one question
you would ask before finalising it.

### Transfer to a different situation

**E14.** A model flags fraudulent transactions with a likelihood ratio of 40. Your prior for a given
transaction is 0.3%. A second, separately trained model also flags it, with a likelihood ratio of 35.
Compute the naive combined posterior, then explain what you would need to know about the two models
before believing it - and what you would expect the true answer to be.

### Explain it to someone non-technical

**E15.** Explain, in under 100 words, why repeating a test is not always the same as getting a second
opinion.

### Optional challenge

**E16.** Bayes' rule extends to more than two hypotheses. A bike fault is one of three kinds - brake,
gear or frame - with prior shares 0.5, 0.3 and 0.2. A diagnostic reports "grinding", which occurs in
70% of brake faults, 40% of gear faults and 5% of frame faults. Compute the posterior over all three,
then add a second symptom and update again. Verify your answer sums to 1 and explain what the
denominator is doing.

In [ ]:
# Your workspace. In memory: table, base_rate, sensitivity, false_positive_rate,
# prior_odds, lr_positive, lr_negative, odds_update, odds_to_probability.

## Mastery check

- [ ] Turn a set of rates into a table of counts and read the posterior off it
- [ ] Name prior, likelihood, evidence and posterior in a problem you have not seen
- [ ] Update in odds form, in your head, for a likelihood ratio you are given
- [ ] Say why a negative result can be far more informative than a positive one
- [ ] Explain when two pieces of evidence may not be multiplied, with an example
- [ ] State how much the prior matters, with two numbers from the same test

## What should now feel instinctive

- Converting rates to counts before doing anything else
- Asking "out of how many?" whenever a test result is described
- Asking what the prior is, and treating "we did not use one" as a claim about it
- Asking *why* a test errs before treating a repeat as independent confirmation
- Noticing that a rare condition makes negatives informative and positives weak

## Flashcards

| Front | Back |
|---|---|
| Bayes' rule | `P(H given E) = P(E given H) x P(H) / P(E)` |
| The evidence term | Everyone who shows this evidence: `P(E\|H)P(H) + P(E\|not H)P(not H)` |
| Odds form | posterior odds = prior odds x likelihood ratio |
| Likelihood ratio | `P(E given H) / P(E given not H)` - how much the evidence favours H |
| The paper method | Invent a population, turn rates into counts, fill four cells, divide |
| Screening example | 1 in 2,000, 99% detection, 1% false positive -> 4.72% |
| Same test, specialist clinic | prior 1 in 3 -> 97.0% |
| Two positives | 83.1% if errors are independent, 4.7% if the error is the person |
| When you may multiply LRs | Only when the evidence is independent given the hypothesis |

## Next

**03-06 · Functions, lines, slopes, logarithms.** Module 03 turns from uncertainty to the shapes that
models are built out of. It is the last piece of groundwork before 03-07's vectors and 03-08's
gradients, which together are everything you need to understand how a model is fitted.